In [0]:


from pyspark.sql.functions import *

print("Starting analytics processing...")


df = spark.table("`for-job-prac-lms`.default.silver_ramen_reviews")

print(f"Total records in silver table: {df.count()}")

# COUNTRY ANALYTICS

country_avg = (
    df.groupBy("country")
    .agg(
        round(avg("stars"), 2).alias("avg_rating"),
        count("*").alias("total_reviews")
    )
    .orderBy(col("avg_rating").desc())
)

# TOP BRANDS ANALYTICS

top_brands = (
    df.groupBy("brand")
    .agg(
        round(avg("stars"), 2).alias("avg_rating"),
        count("*").alias("total_reviews")
    )
    .filter(col("total_reviews") >= 5)
    .orderBy(col("avg_rating").desc())
)

# STYLE ANALYTICS

style_analysis = (
    df.groupBy("style")
    .agg(
        count("*").alias("total_products"),
        round(avg("stars"), 2).alias("avg_rating")
    )
    .orderBy(col("total_products").desc())
)

# WRITE GOLD TABLES

(
    country_avg.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("`for-job-prac-lms`.default.gold_country_ratings")
)

(
    top_brands.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("`for-job-prac-lms`.default.gold_top_brands")
)

(
    style_analysis.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("`for-job-prac-lms`.default.gold_style_analysis")
)

print("Gold tables updated successfully")

print("Available Gold Tables:")

spark.sql("""
SHOW TABLES IN `for-job-prac-lms`.default
""").show(truncate=False)

Starting analytics processing...
Total records in silver table: 1797
Gold tables updated successfully
Available Gold Tables:
+--------+--------------------+-----------+
|database|tableName           |isTemporary|
+--------+--------------------+-----------+
|default |bronze_ramen_reviews|false      |
|default |gold_country_ratings|false      |
|default |gold_style_analysis |false      |
|default |gold_top_brands     |false      |
|default |silver_ramen_reviews|false      |
+--------+--------------------+-----------+

